In [ ]:
!pip install wandb -q


# Experiment Tracking with Weights & Biases (W&B)

Running machine learning experiments without tracking them is like cooking without writing down the recipe.
Every time you tweak a hyperparameter, you should be able to reproduce and compare results.

**W&B** solves this — it automatically logs metrics, model weights, hyperparameters, images, and hardware usage, and visualises everything on an interactive dashboard.

**By the end of this notebook you will:**
- Log training metrics and visualise them on a W&B dashboard
- Track model weights and gradient norms with `wandb.watch`
- Log sample predictions as image tables
- Run a hyperparameter sweep (grid/random/Bayesian)
- Know when to use W&B vs alternatives (TensorBoard, MLflow)


## 1. Setup & Login

Create a free account at [wandb.ai](https://wandb.ai), then get your API key from your profile settings.
Paste it when prompted below.

> **No account?** Use `wandb.init(mode='offline')` to log everything locally without an account.


In [ ]:
import wandb
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# Login — paste your API key when prompted, or use mode='offline'
# wandb.login()
# For offline mode (no account needed):
wandb.setup({'settings': wandb.Settings(mode='offline')})

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('wandb version:', wandb.__version__)


## 2. Instrumenting a Training Loop

The minimal W&B integration requires just 3 lines added to your existing code:

```python
wandb.init(project='my-project', config=hyperparams)   # start a run
wandb.log({'train_loss': loss, 'val_acc': acc})         # inside training loop
wandb.finish()                                          # end of run
```


In [ ]:
# Dataset
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])
train_ds = torchvision.datasets.CIFAR10('./data', train=True,  download=True, transform=tf)
val_ds   = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=tf)
class_names = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

class SimpleCNN(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(128*8*8,256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256,10)
        )
    def forward(self, x): return self.net(x)

def train_with_wandb(config=None):
    # Initialise a W&B run
    with wandb.init(project='mcta4363-deep-learning', config=config, mode='offline') as run:
        cfg = run.config

        train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  num_workers=2)
        val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False, num_workers=2)

        model     = SimpleCNN(dropout=cfg.dropout).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
        criterion = nn.CrossEntropyLoss()

        # wandb.watch logs gradients and parameter histograms every N steps
        wandb.watch(model, log='all', log_freq=50)

        for epoch in range(cfg.epochs):
            # Training
            model.train(); train_loss = correct = total = 0
            for imgs, labels in train_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                optimizer.zero_grad()
                out  = model(imgs)
                loss = criterion(out, labels)
                loss.backward(); optimizer.step()
                train_loss += loss.item(); correct += (out.argmax(1)==labels).sum().item(); total += len(labels)

            # Validation
            model.eval(); val_loss = vcorrect = vtotal = 0
            with torch.no_grad():
                for imgs, labels in val_loader:
                    imgs, labels = imgs.to(device), labels.to(device)
                    out  = model(imgs)
                    loss = criterion(out, labels)
                    val_loss += loss.item(); vcorrect += (out.argmax(1)==labels).sum().item(); vtotal += len(labels)

            metrics = {
                'epoch':      epoch + 1,
                'train_loss': train_loss / len(train_loader),
                'train_acc':  correct / total,
                'val_loss':   val_loss / len(val_loader),
                'val_acc':    vcorrect / vtotal,
                'lr':         optimizer.param_groups[0]['lr'],
            }
            wandb.log(metrics)  # <-- logs to W&B dashboard
            print(f"Epoch {epoch+1} | train_acc={metrics['train_acc']:.4f} | val_acc={metrics['val_acc']:.4f}")

        # Log sample predictions as an image table
        imgs, labels = next(iter(val_loader))
        with torch.no_grad(): preds = model(imgs.to(device)).argmax(1).cpu()
        table = wandb.Table(columns=['image','true','predicted'])
        for i in range(min(8, len(imgs))):
            img_np = (imgs[i].permute(1,2,0).numpy() * 0.5 + 0.5).clip(0,1)
            table.add_data(wandb.Image(img_np), class_names[labels[i]], class_names[preds[i]])
        wandb.log({'predictions': table})

# Run with a config
train_with_wandb(config={
    'epochs': 5, 'batch_size': 128, 'lr': 1e-3,
    'dropout': 0.3, 'weight_decay': 1e-4,
})


## 3. Hyperparameter Sweeps

Instead of manually trying different hyperparameters, W&B Sweeps automate the search.
Define a sweep config, launch an agent, and W&B runs experiments and shows the results on a parallel coordinates plot.


In [ ]:
# Define a sweep over learning rate and dropout
sweep_config = {
    'method': 'grid',   # or 'random', 'bayes'
    'metric': {'name': 'val_acc', 'goal': 'maximize'},
    'parameters': {
        'lr':           {'values': [1e-3, 5e-4]},
        'dropout':      {'values': [0.2, 0.4]},
        'batch_size':   {'value':  128},
        'epochs':       {'value':  3},
        'weight_decay': {'value':  1e-4},
    }
}

print('Sweep config defined.')
print('To run a full sweep (requires W&B account):')
print()
print('  sweep_id = wandb.sweep(sweep_config, project="mcta4363-deep-learning")')
print('  wandb.agent(sweep_id, function=train_with_wandb, count=4)')
print()
print('Running a single sweep trial offline to demonstrate:')
train_with_wandb(config={'epochs':3,'batch_size':128,'lr':5e-4,'dropout':0.2,'weight_decay':1e-4})


## 4. W&B vs Alternatives

| Tool | Best for | Hosted | Free tier |
|------|----------|--------|-----------|
| **W&B** | Teams, sweeps, rich dashboards | Cloud / self-host | Yes (unlimited personal) |
| **TensorBoard** | Quick local visualisation | Local | Always free |
| **MLflow** | Experiment + model registry, self-hosted | Local / cloud | Always free |
| **Comet** | Similar to W&B, academic plans | Cloud | Limited |

**Recommendation for this course:** W&B offline for assignments, W&B cloud for project work.


## 5. Exercises

1. **Dashboard exploration**: Login to your W&B account and view the run dashboard. Try downloading the logged metrics as a CSV. What columns are available?
2. **Add LR scheduler**: Add `CosineAnnealingLR` to `train_with_wandb` and log the learning rate each epoch. Does the W&B dashboard show the decay curve correctly?
3. **Run a sweep**: With a W&B account, run the sweep config above (4 combinations of lr × dropout). Which combination achieves the best `val_acc`? Use the parallel coordinates plot to visualise.
